# Rule-based relation extraction
In this notebook, we will demonstrate how to use a rule-based approach for relation extraction in Latin texts.

Rule-based relation extraction identifies relationships between entities in text using manually crafted patterns, rather than learning from labeled examples. These rules can operate at several levels:
- surface patterns (e.g., regular expressions matching "X, the CEO of Y"),
- syntactic patterns based on dependency parse trees (e.g., extracting a relation when two entities are connected by a specific verb via subject/object dependencies),
- lexico-syntactic patterns that combine keywords with structural constraints (like the classic Hearst patterns for hyponymy: "X such as Y" implies is-a(Y, X)).

---

✅ Advantages of rule-based relation extraction:
- **Interpretability**: Every extraction can be traced back to the exact rule that fired, making it easy to debug and explain results.
- **No training data required**: Rules can be written directly from domain knowledge, which is useful when labeled data is scarce or expensive to produce.
- **High precision on well-defined patterns**: When a domain has consistent, formulaic language (e.g., legal contracts, chemical nomenclature), rules can achieve very high precision.
- **Full control**: Developers can directly encode domain expertise and easily add exceptions or edge cases as they're discovered.
- **Deterministic behavior**: Same input always produces the same output, which matters for auditability and reproducibility.

❌ Disadvantages of rule-based relation extraction:
- **Low recall / poor generalization**: Rules only catch phrasings they were explicitly written for, so they tend to miss the many ways a relation can be expressed in natural language.
- **Labor-intensive**: Writing and maintaining rules for many relation types, especially across domains or languages, requires significant manual effort. For every new relation type, a new set of rules must be crafted and tested.
- **Brittle to linguistic variation**: Passive voice, coordination, coreference, and long-range dependencies often break patterns that were designed for simpler sentence structures.
- **Rule interactions become complex**: As rule sets grow, conflicts and overlaps between rules become hard to manage, and maintenance cost increases superlinearly with rule count.
- **Poor portability**: Rules tuned for one domain (e.g., biomedical text) usually transfer poorly to another (e.g., news text) and need substantial rework.

## step 0: installing Spacy pipelines and LatinCy

In [1]:
!pip install -U spacy==3.8.14
# this version is compatible with LatinCy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# before installing, it is important that you install the Microsoft Visual C++ Build Tools, which is required for compiling some of the dependencies. You can download it from the official Microsoft website.

!pip install "la-core-web-lg @ https://huggingface.co/latincy/la_core_web_lg/resolve/main/la_core_web_lg-3.9.6-py3-none-any.whl"

!pip install "la-core-web-md @ https://huggingface.co/latincy/la_core_web_md/resolve/main/la_core_web_md-3.9.6-py3-none-any.whl"

!pip install "la-core-web-sm @ https://huggingface.co/latincy/la_core_web_sm/resolve/main/la_core_web_sm-3.9.6-py3-none-any.whl"


     ---------------------------------------- 0.0/296.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/296.9 MB ? eta -:--:--
     --------------------------------------- 1.8/296.9 MB 25.0 MB/s eta 0:00:12
      -------------------------------------- 4.2/296.9 MB 14.8 MB/s eta 0:00:20
      -------------------------------------- 7.6/296.9 MB 15.7 MB/s eta 0:00:19
     - ------------------------------------ 10.7/296.9 MB 15.6 MB/s eta 0:00:19
     - ------------------------------------ 15.2/296.9 MB 16.8 MB/s eta 0:00:17
     -- ----------------------------------- 17.3/296.9 MB 15.6 MB/s eta 0:00:18
     -- ----------------------------------- 21.0/296.9 MB 15.8 MB/s eta 0:00:18
     --- ---------------------------------- 26.0/296.9 MB 16.9 MB/s eta 0:00:16
     ---- --------------------------------- 34.6/296.9 MB 19.8 MB/s eta 0:00:14
     ---- --------------------------------- 34.6/296.9 MB 19.8 MB/s eta 0:00:14
     ---- --------------------------------- 34.6/296.9


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/72.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/72.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/72.9 MB ? eta -:--:--
     ---------------------------------------- 0.3/72.9 MB ? eta -:--:--
     -- ------------------------------------- 3.9/72.9 MB 21.3 MB/s eta 0:00:04
     --- ------------------------------------ 6.8/72.9 MB 16.1 MB/s eta 0:00:05
     ---- ----------------------------------- 8.7/72.9 MB 16.3 MB/s eta 0:00:04
     ---- ----------------------------------- 8.7/72.9 MB 16.3 MB/s eta 0:00:04
     ---- ----------------------------------- 8.7/72.9 MB 16.3 MB/s eta 0:00:04
     ---- ----------------------------------- 8.7/72.9 MB 16.3 MB/s eta 0:00:04
     ----- ---------------------------------- 9.4/72.9 MB 6.4 MB/s eta 0:00:10
     ------ --------------------------------- 11.5/72.9 MB 6.9 MB/s eta 0:00:09
     ------- -------------------------------- 13.9/72.9 MB 7.4 MB/s eta 0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
      --------------------------------------- 0.3/19.2 MB ? eta -:--:--
      --------------------------------------- 0.3/19.2 MB ? eta -:--:--
     - ------------------------------------- 0.5/19.2 MB 558.9 kB/s eta 0:00:34
     - ------------------------------------- 0.5/19.2 MB 558.9 kB/s eta 0:00:34
     - ------------------------------------- 0.5/19.2 MB 558.9 kB/s eta 0:00:34
     - ------------------------------------- 0.8/19.2 MB 508.0 kB/s eta 0:00:37
     ----- ---------------------------------- 2.6/19.2 MB 1.7 MB/s eta 0:00:10
     ------------- -------------------------- 6.6/19.2 MB 4.1 MB/s eta 0:00:04
     ------------- -------------------------- 6.6/19.2 MB 4.1 MB/s eta 0:00:04
     ------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# installing additional packages
!pip install pandas
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install tqdm
!pip install tabulate


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## step 1: NER'ed text in spaCy format

For the rule-based relation extraction pipeline we will use spaCy. Thus, it is necessary to have the NER'ed text in a spaCy format.

- do you have NER in SpaCy format?  -> perfect! you can proceed to step 3.
- do you have NER in GLiNER format?  -> no problem! the notebook 'import-export.ipynb' to convert GLiNER output to spaCy format.
- do you have NER in Label Studio format?  -> no problem! go to the notebook 'import-export.ipynb' to convert Label Studio output to spaCy format.
- do you not have any NER'ed text?  -> we got you covered! go to notebook 'NB0-GLiNER2-NER.ipynb' or use our sample text

## step 2: normalise the text

The LatinCy pipeline uses a normalizer to convert the orthography of the text to standardised form. Converting your text to this form before doing anything else is essential for the pipeline to work properly. As Æ and Œ ligatures are automatically converted to AE and OE. This would be problematic as the span of a word changes and the entities would not be aligned properly.

The following code will convert your text to the standardised form. There is also the possibility withing the code to remove any bracketed text, which is often used to indicate editorial additions or corrections. However, it is important to note that this may also remove important information from the text, so use this option with caution.

In [1]:
import json
import re
from pathlib import Path

LIGATURE_MAP = {
    "v": "u", "V": "U",
    "j": "i", "J": "I",
    "æ": "ae", "Æ": "Ae",
    "œ": "oe", "Œ": "Oe",
}

def find_bracket_spans(text, pattern=r"\[Col\.[^\]]*\]"):
    """Editorial apparatus insertions like [Col. 0205E], number-only brackets excluded."""
    return [(m.start(), m.end()) for m in re.finditer(pattern, text)]

def normalize_text_and_build_map(text, strip_brackets=True, collapse_whitespace=True):
    remove_mask = [False] * len(text)
    if strip_brackets:
        for s, e in find_bracket_spans(text):
            for i in range(s, e):
                remove_mask[i] = True

    out_chars = []
    offset_map = []
    running_len = 0
    prev_emitted_is_space = True

    for i, ch in enumerate(text):
        offset_map.append(running_len)
        replacement = "" if remove_mask[i] else LIGATURE_MAP.get(ch, ch)
        if collapse_whitespace and replacement == " " and prev_emitted_is_space:
            replacement = ""
        if replacement:
            out_chars.append(replacement)
            running_len += len(replacement)
            prev_emitted_is_space = replacement[-1] == " "

    offset_map.append(running_len)
    normalized_text = "".join(out_chars)
    return normalized_text, offset_map

In [2]:
def normalize_entities(entities, offset_map, normalized_text):
    problems = []
    new_entities = []
    for ent in entities:
        start, end = ent["start"], ent["end"]
        if start >= len(offset_map) or end >= len(offset_map):
            problems.append((ent, "start/end out of range"))
            continue
        new_start, new_end = offset_map[start], offset_map[end]
        if new_end <= new_start and end > start:
            problems.append((ent, "entity collapsed — was inside removed bracket text"))
            continue
        new_ent = dict(ent)
        new_ent["start"] = new_start
        new_ent["end"] = new_end
        new_ent["text"] = normalized_text[new_start:new_end]
        new_ent["orig_start"] = start
        new_ent["orig_end"] = end
        new_ent["orig_text"] = ent.get("text")
        new_entities.append(new_ent)
    return new_entities, problems

In [3]:
def normalize_document(doc):
    text = doc["text"]
    normalized_text, offset_map = normalize_text_and_build_map(text)
    new_entities, problems = normalize_entities(doc.get("entities", []), offset_map, normalized_text)
    new_doc = dict(doc)
    new_doc["text"] = normalized_text
    new_doc["entities"] = new_entities
    return new_doc, problems

def normalize_json_data(data):
    new_data, all_problems = [], []
    for i, doc in enumerate(data):
        new_doc, problems = normalize_document(doc)
        new_data.append(new_doc)
        for ent, issue in problems:
            all_problems.append((i, ent, issue))
    return new_data, all_problems

def normalize_file(in_path: Path, out_path: Path):
    data = json.loads(in_path.read_text(encoding="utf-8"))

    was_single_doc = isinstance(data, dict)
    docs = [data] if was_single_doc else data

    new_data, problems = normalize_json_data(docs)

    output = new_data[0] if was_single_doc else new_data
    out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")

    n_ents = sum(len(d.get("entities", [])) for d in new_data)
    print(f"{in_path.name} -> {out_path.name}: {len(new_data)} docs, {n_ents} entities remapped")
    if problems:
        print(f"  ⚠ {len(problems)} problem(s):")
        for doc_i, ent, issue in problems:
            print(f"    doc[{doc_i}] id={ent.get('id')!r} {ent.get('text')!r} — {issue}")
    return problems

In [5]:
# --- CHANGE THIS to your file or folder ---
path = Path("sample_texts/hagio_full2_spacy_rels-is_located_at")

total_problems = 0

if path.is_file():
    out_path = path.with_name(path.stem + "_norm" + path.suffix)
    total_problems += len(normalize_file(path, out_path))

elif path.is_dir():
    out_dir = path.with_name(path.name + "_norm")
    out_dir.mkdir(exist_ok=True)
    json_files = sorted(path.glob("*.json"))
    if not json_files:
        print(f"No .json files found in {path}")
    for f in json_files:
        out_file = out_dir / (f.stem + "_norm" + f.suffix)
        total_problems += len(normalize_file(f, out_file))
    print(f"\nAll normalized files written to: {out_dir}")

print(f"\n{'⚠ ' + str(total_problems) + ' total problem(s) — review above.' if total_problems else 'No remapping problems detected.'}")

hagio_full2_task_21197.json -> hagio_full2_task_21197_norm.json: 1 docs, 15 entities remapped
hagio_full2_task_21199.json -> hagio_full2_task_21199_norm.json: 1 docs, 15 entities remapped
hagio_full2_task_21203.json -> hagio_full2_task_21203_norm.json: 1 docs, 36 entities remapped
hagio_full2_task_21204.json -> hagio_full2_task_21204_norm.json: 1 docs, 26 entities remapped

All normalized files written to: sample_texts\hagio_full2_spacy_rels-is_located_at_norm

No remapping problems detected.


## step 3: define the patterns for the relation extraction

To do rule-based relation extraction, you first need to define the patterns that indicate a relation between two entities. If you have already defined the patterns, you can skip this step and load the patterns directly. If not, the following code blocks will help you to define the patterns.

Visualising the entities and their dependencies in the text can help you identify what patterns spaCy can identify. The following code block will visualize the entities and their dependencies in the text. You can change the sentence number to visualise different sentences in you specific text.

recommended steps for defining patterns:
1. visualise the entities and dependencies in the text
2. look for patterns in the entities and dependencies
3. give your sample text and your detected entities to either a human annotator or an LLM to trial your patterns
4. run the relation extraction code with your patterns and check the results
5. adjust and try again until you are satisfied with the results


### 3.1 visualise the entities

In [5]:
# displaCy visualization

import json
from pathlib import Path
import spacy
from spacy import displacy

# --- Config (self-contained) ---
TEXT_PATH = Path("sample_texts/rule-based-example.txt")
NER_PATH = Path("sample_texts/rule-based-example_NER.json")
MODEL_NAME = "la_core_web_lg"

# --- Safety checks ---
if not TEXT_PATH.exists():
    raise FileNotFoundError(f"Missing text file: {TEXT_PATH}")
if not NER_PATH.exists():
    raise FileNotFoundError(f"Missing NER file: {NER_PATH}")

# --- Load inputs ---
text = TEXT_PATH.read_text(encoding="utf-8")
with NER_PATH.open("r", encoding="utf-8") as f:
    gliner_results = json.load(f)

# --- Build spaCy doc from GLiNER output ---
# nlp = spacy.load("la_core_web_lg")   # <- causes orthographic normalization
nlp = spacy.blank("la")                # <- tokenizer only, text stays untouched


def build_spacy_doc(text, gliner_json):
    doc = nlp(text)
    spans = []
    problems = []

    for entity_group in gliner_json.get("entities", []):
        for label, mentions in entity_group.items():
            for mention in mentions:
                start, end, expected = mention["start"], mention["end"], mention["text"]
                span = doc.char_span(start, end, label=label, alignment_mode="expand")
                if span is None:
                    problems.append((label, expected, "char_span returned None"))
                    continue
                if span.text != expected:
                    problems.append((label, expected, f"got {span.text!r} instead"))
                    continue
                spans.append(span)

    if problems:
        print(f"⚠ {len(problems)} entity/token misalignments:")
        for label, expected, issue in problems:
            print(f"  [{label}] expected {expected!r} — {issue}")

    spans = sorted(spans, key=lambda s: (s.start, -(s.end - s.start)))
    filtered = []
    last_end = -1
    for span in spans:
        if span.start >= last_end:
            filtered.append(span)
            last_end = span.end

    doc.ents = filtered
    return doc


doc = build_spacy_doc(text, gliner_results)

colors = {
    "person": "#ffb703",
    "group": "#8ecae6",
    "object": "#adb5bd",
    "divine_entity": "#e63946",
    "place": "#90be6d",
    "institution": "#f1faee",
    "pronoun": "#f4a261",
}
displacy.render(doc, style="ent", jupyter=True, options={"colors": colors})

### 3.2 visualise the dependencies
In the SpaCy pipeline, the dependency parser is responsible for identifying the grammatical relationships between words in a sentence. Visualizing these dependencies can help you understand how entities are connected and how to define patterns for relation extraction.

In [6]:
import spacy
from spacy import displacy

nlp = spacy.load("la_core_web_lg")

with open("sample_texts/rule-based-example.txt", "r", encoding="utf-8") as f:
    text = f.read()

doc = nlp(text)

# Choose the sentence you want to display
sentence_number = 0  # change this to 0, 1, 2, ...
sentences = list(doc.sents)

if sentence_number < 0 or sentence_number >= len(sentences):
    raise IndexError(f"sentence_number must be between 0 and {len(sentences) - 1}")

sent_doc = sentences[sentence_number]

displacy.render(sent_doc.as_doc(), style="dep", options={"compact": True, "distance": 100})

## step 4: run the relation extraction code

Now that we have visualised the entities and dependencies in the text, we can define the patterns for the relation extraction. The following code blocks will help you to define the patterns and run the relation extraction code.

⚠️⚠️ important notes: from this point on, the code is designed to work with the spaCy format. If you have a GLiNER or Label Studio output, please convert it to the spaCy format using the provided 'import-export.ipynb' notebook before proceeding. The spaCy format is a tuple of (text, {"entities": [(start, end, label), ...]}), which is compatible with the relation extraction code below.

In [20]:
# Standalone: load spaCy training tuple JSON and build docs with entities
# Handles a single JSON file OR a folder of JSON files, and files that
# contain either a single record (dict) or a list of records.

import json
from pathlib import Path
import spacy

SPACY_TUPLE_PATH = Path("sample_texts/relation-annotated-example_spacy_rels-is_located_at")  # file OR folder
base_name_spacy_tuple = Path(SPACY_TUPLE_PATH.stem)

MODEL_NAME = "la_core_web_lg"

nlp = spacy.load(MODEL_NAME)


def load_records(path: Path):
    """
    Load one or more spaCy-format records from `path`.
    - If path is a file: read that one file.
    - If path is a folder: read every *.json file inside it.
    Each file may itself contain a single record ({"text": ..., "entities": ...})
    or a list of records — both are normalized into one flat list here.
    Returns a list of (source_file, entry) tuples so you always know which
    file each entry came from.
    """
    if path.is_file():
        json_files = [path]
    elif path.is_dir():
        json_files = sorted(path.glob("*.json"))
        if not json_files:
            raise FileNotFoundError(f"No .json files found in folder: {path}")
    else:
        raise FileNotFoundError(f"Path not found: {path}")

    records = []
    for fp in json_files:
        with fp.open("r", encoding="utf-8") as f:
            data = json.load(f)
        data_list = [data] if isinstance(data, dict) else data
        for entry in data_list:
            records.append((fp, entry))
    return records


def build_doc(entry, nlp):
    """Build one spaCy Doc with entities from a single record."""
    text = entry["text"]
    entities_raw = [(e["start"], e["end"], e["label"]) for e in entry["entities"]]

    doc = nlp.make_doc(text)
    for name, proc in nlp.pipeline:
        if name == "ner":
            continue
        doc = proc(doc)

    spans = []
    for start, end, label in entities_raw:
        span = doc.char_span(start, end, label=label, alignment_mode="expand")
        if span is None:
            print(f"  Could not align [{start}:{end}]: {text[start:end]}")
        else:
            spans.append(span)

    # Filter overlaps at token level (longest-first)
    spans.sort(key=lambda s: (s.start, -(s.end - s.start)))
    filtered_spans = []
    for span in spans:
        overlaps = any(
            not (span.end <= ex.start or span.start >= ex.end) for ex in filtered_spans
        )
        if not overlaps:
            filtered_spans.append(span)

    doc.set_ents(filtered_spans)
    return doc, len(spans), len(filtered_spans)


# --- Process everything ---
records = load_records(SPACY_TUPLE_PATH)
print(f"Loaded {len(records)} record(s) from {SPACY_TUPLE_PATH}")

docs = []  # keep built docs around for downstream use (relation extraction, etc.)
for idx, (source_file, entry) in enumerate(records):
    print(f"\n--- Processing entry {idx + 1} ({source_file.name}) ---")
    doc, n_spans, n_filtered = build_doc(entry, nlp)
    docs.append({"source_file": source_file, "entry": entry, "doc": doc})
    print(f"  Loaded {n_filtered} entities into doc (filtered from {n_spans})")

Loaded 4 record(s) from sample_texts\relation-annotated-example_spacy_rels-is_located_at

--- Processing entry 1 (relation-annotated-example_task_21197.json) ---
  Loaded 14 entities into doc (filtered from 15)

--- Processing entry 2 (relation-annotated-example_task_21199.json) ---
  Loaded 15 entities into doc (filtered from 15)

--- Processing entry 3 (relation-annotated-example_task_21203.json) ---
  Loaded 33 entities into doc (filtered from 36)

--- Processing entry 4 (relation-annotated-example_task_21204.json) ---
  Loaded 23 entities into doc (filtered from 26)


In [21]:
# Keep a clean copy of doc before merging entities — retokenization is destructive
import copy

# Retokenize the original doc directly (merges happen in-place)
with doc.retokenize() as retokenizer:
    for ent in doc.ents:
        retokenizer.merge(ent, attrs={"ENT_TYPE": ent.label_})

# Now make a deepcopy if you need to preserve this state for later
parsed_doc = copy.deepcopy(doc)

# sanity check: confirm the entity labels GLiNER actually produced
print(sorted({t.ent_type_ for t in doc if t.ent_type_}))

['group', 'institution', 'object', 'person', 'place', 'pronoun']


In [22]:
# -------------------------------------------------------------------------------------------------
# this is the pattern for the 'is_located_at'-relation
# -------------------------------------------------------------------------------------------------

from spacy.matcher import DependencyMatcher

LOCATABLE = {"place", "institution", "object"}  # confirm against your actual GLiNER labels

matcher = DependencyMatcher(parsed_doc.vocab)


# shared building block: tail entity must be governed by a preposition (ADP)
def with_adp_check(head_rel_op, mediator_attrs=None):
    """Builds a pattern requiring head -> [mediator ->] tail -> ADP child."""
    pattern = [
        {"RIGHT_ID": "head", "RIGHT_ATTRS": {"ENT_TYPE": {"IN": list(LOCATABLE)}}},
    ]
    if mediator_attrs is None:
        pattern.append(
            {"LEFT_ID": "head", "REL_OP": head_rel_op, "RIGHT_ID": "tail",
             "RIGHT_ATTRS": {"ENT_TYPE": {"IN": ["place"]}}}
        )
    else:
        pattern.append(
            {"LEFT_ID": "head", "REL_OP": head_rel_op, "RIGHT_ID": "mediator",
             "RIGHT_ATTRS": mediator_attrs}
        )
        pattern.append(
            {"LEFT_ID": "mediator", "REL_OP": ">", "RIGHT_ID": "tail",
             "RIGHT_ATTRS": {"ENT_TYPE": {"IN": ["place"]}}}
        )
    pattern.append(
        {"LEFT_ID": "tail", "REL_OP": ">", "RIGHT_ID": "prep",
         "RIGHT_ATTRS": {"POS": "ADP"}}
    )
    return pattern


# A — direct: head -> tail, tail governed by ADP   (monasterio -> flumen)
pattern_direct = with_adp_check(">")

# B — participle-mediated: head -> participle -> tail   (ecclesia -> colle; xenodochium -> portam civitatis)
pattern_participle = with_adp_check(
    ">", mediator_attrs={"DEP": {"IN": ["acl", "amod"]},
                         "MORPH": {"IS_SUPERSET": ["VerbForm=Part"]}}
)

# C — verb-mediated: head and tail are co-arguments of a verb   (oratorium -> radices silvae; arca reliquiarum -> altare)
pattern_verb = [
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "head",
     "RIGHT_ATTRS": {"ENT_TYPE": {"IN": list(LOCATABLE)},
                     "DEP": {"IN": ["nsubj", "nsubj:pass", "obj", "obl"]}}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "tail",
     "RIGHT_ATTRS": {"ENT_TYPE": {"IN": ["place"]}}},
    {"LEFT_ID": "tail", "REL_OP": ">", "RIGHT_ID": "prep",
     "RIGHT_ATTRS": {"POS": "ADP"}},
]


matcher.add("is_located_at_direct", [pattern_direct])
matcher.add("is_located_at_participle", [pattern_participle])
matcher.add("is_located_at_verb", [pattern_verb])


In [24]:

# Produce only the full JSON of traced relations (one file: <run_base>_rel_full.json)
import copy
import json
from pathlib import Path
from spacy.matcher import DependencyMatcher
import pandas as pd

# run_base: use file stem for single-file input, or folder name for batch
run_base = SPACY_TUPLE_PATH.stem if SPACY_TUPLE_PATH.is_file() else SPACY_TUPLE_PATH.name

# Build matcher once using nlp.vocab
matcher = DependencyMatcher(nlp.vocab)
matcher.add("is_located_at_direct", [pattern_direct])
matcher.add("is_located_at_participle", [pattern_participle])
matcher.add("is_located_at_verb", [pattern_verb])



all_rows = []
parsed_doc = None  # will hold the last parsed doc (keeps compatibility with visualisation code)

for i_doc, doc_entry in enumerate(docs, start=1):
    source_file = doc_entry["source_file"]
    doc = doc_entry["doc"]

    # Merge entity spans (retokenize) for this doc
    with doc.retokenize() as retokenizer:
        for ent in doc.ents:
            retokenizer.merge(ent, attrs={"ENT_TYPE": ent.label_})

    parsed_doc = copy.deepcopy(doc)  # keep the last parsed_doc for downstream visualisation

    # Run matcher on this parsed_doc
    matches = matcher(parsed_doc)
    for match_id, token_ids in matches:
        match_label = parsed_doc.vocab.strings[match_id]

        span_start = min(token_ids)
        span_end = max(token_ids) + 1
        match_span = parsed_doc[span_start:span_end]
        context_sentence = match_span.sent.text.strip()

        # find entity tokens among matched indices
        entity_tokens = [parsed_doc[i] for i in token_ids if parsed_doc[i].ent_type_]
        if len(entity_tokens) != 2:
            continue

        head, tail = entity_tokens[0], entity_tokens[1]
        if head.i == tail.i:
            continue



        all_rows.append({
            "source_file": source_file.name,
            "source_path": str(source_file),
            "head_i": int(head.i),
            "tail_i": int(tail.i),
            "head": head.text,
            "head_type": head.ent_type_,
            "tail": tail.text,
            "tail_type": tail.ent_type_,
            "relation": match_label,
            "context": context_sentence,
        })

# Build DataFrame across all docs
df = pd.DataFrame(all_rows)

if df.empty:
    print("No relations found by the matcher.")
else:
    # compute symmetric / pair_key
    pair_set = set(zip(df["head_i"], df["tail_i"]))
    df["symmetric"] = df.apply(lambda r: (r["tail_i"], r["head_i"]) in pair_set, axis=1)
    df["pair_key"] = df.apply(lambda r: tuple(sorted((int(r["head_i"]), int(r["tail_i"])))), axis=1)

    # Prepare JSON records (full info)
    full_json_records = df[[
        "source_file", "source_path", "head_i", "tail_i",
        "head", "head_type", "tail", "tail_type",
        "relation", "context", "symmetric", "pair_key"
    ]].to_dict(orient="records")

    # Ensure output directory exists and write JSON
    out_dir = Path("relations")
    out_dir.mkdir(exist_ok=True)
    out_full_json = out_dir / f"{run_base}_rel_full.json"
    with open(out_full_json, "w", encoding="utf-8") as jf:
        json.dump(full_json_records, jf, ensure_ascii=False, indent=2)

    print(f"Saved full relations JSON: {out_full_json} ({len(full_json_records)} entries)")


Saved full relations JSON: relations\relation-annotated-example_spacy_rels-is_located_at_rel_full.json (1 entries)


In [25]:
# Load the produced full JSON and display a CSV-preview (DataFrame). This is a preview only (not saved).
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

run_base = SPACY_TUPLE_PATH.stem if SPACY_TUPLE_PATH.is_file() else SPACY_TUPLE_PATH.name
in_file = Path("relations") / f"{run_base}_rel_full.json"

if not in_file.exists():
    print(f"Preview JSON not found: {in_file}")
else:
    with open(in_file, "r", encoding="utf-8") as f:
        records = json.load(f)

    if not records:
        print("JSON contains no relation records.")
    else:
        # Build DataFrame preview
        df_preview = pd.DataFrame(records)
        # reorder / pick preview columns
        preview_cols = [ "head", "head_type", "tail", "tail_type", "relation", "context"]
        df_preview = df_preview[[c for c in preview_cols if c in df_preview.columns]]

        display(Markdown(f"**Preview of relations from:** `{in_file}`  \n(rows shown: {len(df_preview)})"))
        display(df_preview.head(200))  # notebook preview only

        # Also set `rows` for downstream visualisation (if needed)
        rows = [
            {
                "head_i": int(r.get("head_i")),
                "tail_i": int(r.get("tail_i")),
                "head": r.get("head"),
                "head_type": r.get("head_type"),
                "tail": r.get("tail"),
                "tail_type": r.get("tail_type"),
                "relation": r.get("relation"),
                "context": r.get("context"),
            }
            for r in records
        ]
        # Note: `parsed_doc` remains the last parsed doc processed by Cell A (used by visualization).
        print("Preview prepared (not saved).")

**Preview of relations from:** `relations\relation-annotated-example_spacy_rels-is_located_at_rel_full.json`  
(rows shown: 1)

,head,head_type,tail,tail_type,relation,context
0,corpus,object,loco,place,is_located_at_verb,[14] In ipso quoque loco non habebatur tunc te...


Preview prepared (not saved).
